# SI4006 · Sesión 5 — Lab: **Métricas de evaluación**  ·  SOLUCIONES

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 2 — Evaluación

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

En M1 midieron su modelo con una métrica simple. Hoy ven, en números, dónde fallan las métricas
clásicas, qué significa la *perplexity*, y cómo se ve un benchmark real por dentro. El harness
calificado (entrega M2) se construye en **S06**; este lab sirve para entender las piezas.

> **No necesitan GPU.** Todo corre en CPU en un par de minutos. Si tienen T4 activa, mejor.

## 0 · Setup

Colab 2026 ya trae `transformers` y `torch` (5.x). **No los fijamos.** Solo instalamos las librerías
de métricas: `evaluate` (envuelve BLEU/ROUGE), `sacrebleu`, `rouge_score`, y `sentence-transformers`
para la similitud por embeddings.

In [1]:
# Instalamos SOLO lo que falta. No fijamos transformers/torch (usamos los de Colab).
%pip install -q evaluate sacrebleu rouge_score sentence-transformers
print('\nListo.')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 2.6 MB/s eta 0:00:00

Listo.


In [2]:
import torch, transformers
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('transformers', transformers.__version__, '| torch', torch.__version__, '| device:', device)

transformers 5.13.1 | torch 2.11.0+cpu | device: cpu


---
# Lab A · Las métricas clásicas, y dónde fallan

En este lab le damos a las métricas **una referencia** (la respuesta correcta) y **tres candidatos**.
Antes de correr cada celda, **predigan en su grupo** qué va a pasar; luego lo comprueban.

| Candidato | Cómo es | Qué esperar |
|---|---|---|
| **1 · Casi exacta** | mismas palabras, un cambio mínimo | todas las métricas altas |
| **2 · Paráfrasis perfecta** | mismo significado, **otras palabras** | BLEU/ROUGE muy bajos; embeddings alto |
| **3 · Fluido pero falso** | suena bien, **significado distinto** | ninguna métrica, sola, lo detecta |

In [3]:
# La referencia (lo que consideramos la respuesta correcta) y tres candidatos.
referencia = 'El gato duerme sobre la alfombra roja.'

candidatos = {
    '1 · casi exacta        ': 'El gato dormía sobre la alfombra roja.',       # cambia un verbo
    '2 · paráfrasis perfecta': 'Sobre el tapete rojo descansa el felino.',     # mismo sentido, otras palabras
    '3 · fluido pero falso  ': 'El perro corre por el jardín verde.',          # suena bien, otro significado
}
print('REFERENCIA:', referencia)
for n, c in candidatos.items():
    print(f'  {n} -> {c}')

REFERENCIA: El gato duerme sobre la alfombra roja.
  1 · casi exacta         -> El gato dormía sobre la alfombra roja.
  2 · paráfrasis perfecta -> Sobre el tapete rojo descansa el felino.
  3 · fluido pero falso   -> El perro corre por el jardín verde.


### A.1 · BLEU y ROUGE — solapamiento de palabras

`BLEU` mira cuántos n-gramas del candidato aparecen en la referencia (precisión); `ROUGE` mira
cuántos de la referencia recupera el candidato (recall). Ambos **cuentan palabras, no significado**.


In [4]:
import evaluate
sacrebleu = evaluate.load('sacrebleu')   # BLEU, escala 0-100
rouge     = evaluate.load('rouge')       # ROUGE, escala 0-1

print(f'{"candidato":<26} {"BLEU":>7} {"ROUGE-L":>9}')
for n, cand in candidatos.items():
    bleu = sacrebleu.compute(predictions=[cand], references=[[referencia]])['score']
    rgL  = rouge.compute(predictions=[cand], references=[referencia])['rougeL']
    print(f'{n:<26} {bleu:>7.1f} {rgL:>9.2f}')

candidato                     BLEU   ROUGE-L
1 · casi exacta               59.5      0.80
2 · paráfrasis perfecta        5.5      0.14
3 · fluido pero falso          6.6      0.13


> **✅ Salida esperada (aprox., verificada):**
> ```
> candidato                     BLEU   ROUGE-L
> 1 · casi exacta               59.5      0.80
> 2 · paráfrasis perfecta        5.5      0.14
> 3 · fluido pero falso          6.6      0.13
> ```
> La **paráfrasis perfecta** (candidato 2) se desploma a BLEU 5.5 / ROUGE 0.14 **aunque significa
> exactamente lo mismo** que la referencia: casi no comparte palabras. Peor aún: el candidato 3
> (*fluido pero falso*, significado equivocado) saca **más** BLEU que la paráfrasis correcta, solo
> porque repite palabras vacías (“el”, “por”). **Las n-grama premiaron la respuesta equivocada.**

### A.2 · Similitud por embeddings — medir significado

Ahora medimos **sentido**, no palabras: convertimos cada texto en un embedding y comparamos con
**coseno** (la misma idea del RAG de S01 y del “banco” de S02). Un modelo multilingüe entiende que
*“felino”* ≈ *“gato”* y *“tapete”* ≈ *“alfombra”*.


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Modelo de embeddings multilingüe, pequeño (~470 MB). Para inglés sirve igual.
st = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

def similitud(a, b):
    ea, eb = st.encode([a, b])
    return float(np.dot(ea, eb) / (np.linalg.norm(ea) * np.linalg.norm(eb)))

print(f'{"candidato":<26} {"coseno":>7}')
for n, cand in candidatos.items():
    print(f'{n:<26} {similitud(referencia, cand):>7.2f}')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

candidato                   coseno
1 · casi exacta               0.99
2 · paráfrasis perfecta       0.88
3 · fluido pero falso         0.01


> **✅ Salida esperada (aprox., verificada):**
> ```
> candidato                    coseno
> 1 · casi exacta               0.99
> 2 · paráfrasis perfecta       0.88
> 3 · fluido pero falso         0.01
> ```
> El embedding **rescata la paráfrasis** (candidato 2 = 0.88) que BLEU había mandado al piso, y
> hunde al candidato 3 (0.01, significado distinto). Justo al revés que BLEU: aquí el orden **sí**
> refleja el significado. Mide sentido, no coincidencia de palabras.

### A.3 · La lectura

Pongan las dos tablas (BLEU/ROUGE y coseno) una al lado de la otra:

- **Candidato 1 (casi exacta):** todo alto. El caso fácil, donde cualquier métrica funciona.
- **Candidato 2 (paráfrasis):** BLEU/ROUGE muy bajos (más bajos, incluso, que el candidato 3),
  pero coseno alto. Las métricas n-grama castigan un acierto y hasta premian un error.
- **Candidato 3 (fluido pero falso):** coseno cercano a 0 (correcto), pero suena bien. Una métrica
  de fluidez (perplexity, siguiente sección) lo aprobaría. Ninguna métrica, por sí sola, lo detecta.

> **Conclusión del Lab A:** una sola métrica siempre se puede engañar. Por eso el harness de M2
> combina varias dimensiones. Se arma en S06.

---
# Perplexity · ¿qué tan sorprendido está el modelo?

`perplexity = exp(cross-entropy)`. Baja = el modelo predice bien el texto que ve (no se “sorprende”).
No necesita respuesta de referencia: solo el texto y un modelo. La medimos con un modelo pequeño
comparando una frase **fluida** contra la **misma frase desordenada**.

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, math

PPL_MODEL = 'distilgpt2'   # pequeño y rápido. Cámbienlo por su modelo (ej. Qwen2.5-0.5B) para su idioma.
ppl_tok = AutoTokenizer.from_pretrained(PPL_MODEL)
ppl_model = AutoModelForCausalLM.from_pretrained(PPL_MODEL).to(device).eval()

def perplexity(texto):
    ids = ppl_tok(texto, return_tensors='pt').input_ids.to(device)
    with torch.no_grad():
        loss = ppl_model(ids, labels=ids).loss   # cross-entropy media
    return math.exp(loss.item())

fluida     = 'The sun rises in the east every single morning.'
desordenada = 'Morning east the every single sun rises in the.'
print(f'fluida       -> perplexity {perplexity(fluida):8.1f}')
print(f'desordenada  -> perplexity {perplexity(desordenada):8.1f}')

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


fluida       -> perplexity    119.1
desordenada  -> perplexity   1672.5


> **✅ Salida esperada (verificada):** la frase **fluida** ≈ **119** de perplexity; la **desordenada**
> ≈ **1670**. El modelo predice bien lo fluido y se “sorprende” con lo desordenado. El orden importa —
> es la señal que capta. (Los valores absolutos dependen del modelo/tokenizer; lo comparable es el
> contraste entre las dos frases con el mismo modelo.)

**La letra pequeña de perplexity:**
- Solo comparable con **el mismo modelo y el mismo tokenizer** sobre el mismo texto.
- **No** es comparable entre datasets o idiomas distintos (son escalas diferentes).
- Mide **fluidez / probabilidad**, no si la respuesta es correcta, veraz o útil. El candidato 3 del
  Lab A (*fluido pero falso*) tendría perplexity baja — y aun así está mal.

---
# Lab B · Un benchmark real, por dentro

Los leaderboards salen de **benchmarks**: datasets con respuestas fijas y una regla de puntuación.
Abrimos **MMLU** (conocimiento en 57 materias, opción múltiple) y lo miramos por dentro para no
creerle a ciegas a un “85%”.

> Nota de Colab: `load_dataset` exige el id **con namespace** (`cais/mmlu`) y un **subject** como
> configuración. Usamos uno pequeño.

In [7]:
from datasets import load_dataset

# Un subject cualquiera de MMLU. 'test' trae las preguntas con su respuesta correcta.
mmlu = load_dataset('cais/mmlu', 'high_school_computer_science', split='test')
print(mmlu)
print('\nTotal de preguntas en este subject:', len(mmlu))

README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

high_school_computer_science/test-00000-(…): reconstructing file:   0%|          |  0.00B / 27.3kB            

high_school_computer_science/test-00000-(…): downloading bytes:           |  0.00B            

high_school_computer_science/validation-(…): reconstructing file:   0%|          |  0.00B / 5.28kB            

...

high_school_computer_science/dev-00000-o(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/9 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'subject', 'choices', 'answer'],
    num_rows: 100
})

Total de preguntas en este subject: 100


In [8]:
# Miremos 2 preguntas por dentro: enunciado, opciones y la respuesta correcta.
letras = ['A', 'B', 'C', 'D']
for i in range(2):
    ej = mmlu[i]
    print('PREGUNTA:', ej['question'])
    for k, op in enumerate(ej['choices']):
        print(f'   {letras[k]}) {op}')
    print('   RESPUESTA CORRECTA:', letras[ej['answer']])
    print('-' * 70)

PREGUNTA: Let x = 1. What is x << 3 in Python 3?
   A) 1
   B) 3
   C) 8
   D) 16
   RESPUESTA CORRECTA: C
----------------------------------------------------------------------
PREGUNTA: In Python 3, which of the following function convert a string to an int in python?
   A) int(x [,base])
   B) long(x [,base] )
   C) float(x)
   D) str(x)
   RESPUESTA CORRECTA: A
----------------------------------------------------------------------


### B.1 · La regla de puntuación: exact-match

Puntuar MMLU es simple: el modelo elige una letra y se compara con la correcta. **Accuracy** = fracción
de aciertos. Nada de significado ni matices: coincide o no coincide.

In [9]:
# Simulamos las respuestas de un 'modelo' para ver cómo se puntúa (exact-match).
gold      = [mmlu[i]['answer'] for i in range(5)]          # respuestas correctas (índices 0-3)
respuestas_modelo = [0, 2, 1, gold[3], gold[4]]            # dos primeras a propósito pueden fallar

aciertos = sum(1 for g, r in zip(gold, respuestas_modelo) if g == r)
print('gold           :', [letras[g] for g in gold])
print('modelo         :', [letras[r] for r in respuestas_modelo])
print(f'accuracy (5 preg): {aciertos}/5 = {aciertos/5:.2f}')

gold           : ['C', 'A', 'A', 'C', 'B']
modelo         : ['A', 'C', 'B', 'C', 'B']
accuracy (5 preg): 2/5 = 0.40


### B.2 · Por qué desconfiar de un puntaje alto

Piénsenlo con lo que vieron en las slides:

- **Contaminación:** estas preguntas están por todo internet. Si el modelo las vio al entrenar, las
  *“sabe”* sin razonar — el puntaje sube sin que el modelo sea mejor.
- **Mide UNA cosa:** conocimiento de opción múltiple. No mide si sirve para **su** dominio.
- **Se sobre-ajusta:** se puede *tunear* el prompt para este formato y no trasladarse a nada real.

> **Conclusión del Lab B:** un benchmark es cómodo y comparable, pero ninguno mide su dominio. Por eso
> el paso siguiente es construir **su propio eval set**.

---
# Su tarea para S06 · la semilla de su eval set

Aquí empieza su entrega **M2**. Traigan **diez ejemplos *gold*** de su dominio: cada uno es un par
`input → salida esperada` (o, si su tarea es abierta, una nota de qué haría *buena* a la respuesta).
En S06 los convierten en un **harness ejecutable** que produce el scorecard de su baseline.

Completen la plantilla de abajo (hay 3 de ejemplo del dominio *educación*; lleguen a 10 con los suyos).

In [10]:
# Plantilla del eval set semilla. Reemplacen por los ejemplos de SU dominio y lleguen a 10.
eval_set = [
    {'input': 'Explica qué es una fracción.',
     'esperado': 'Una fracción representa partes de un todo; por ejemplo 1/4 es una de cuatro partes iguales.',
     'criterio': 'correcta, clara y apropiada para un estudiante de secundaria'},

    {'input': '¿Qué es un número primo?',
     'esperado': 'Un número que solo se divide exactamente entre 1 y él mismo (2, 3, 5, 7, ...).',
     'criterio': 'definición correcta + al menos un ejemplo'},

    {'input': 'Explica la suma de fracciones con igual denominador.',
     'esperado': 'Se suman los numeradores y se mantiene el denominador.',
     'criterio': 'regla correcta, sin pasos de más'},

    # TODO — añadan 7 ejemplos más de su dominio (input + esperado + criterio).
    # Incluyan al menos 2 casos DIFÍCILES o de borde (ambiguos, donde el modelo suele fallar).
]

print(f'Ejemplos en el eval set: {len(eval_set)} / 10')
assert all('input' in e and 'esperado' in e for e in eval_set), 'Cada ejemplo necesita input y esperado.'
print('Formato OK. Guárdenlo en el repo del equipo — es la semilla de M2.')

Ejemplos en el eval set: 3 / 10
Formato OK. Guárdenlo en el repo del equipo — es la semilla de M2.


---
### Lo que se llevan de hoy
1. Las métricas n-grama (BLEU/ROUGE) **castigan paráfrasis válidas**; los embeddings miden significado.
2. **Perplexity** mide fluidez, no utilidad ni verdad.
3. Los **benchmarks** son cómodos pero miden una cosa y se contaminan; ninguno mide su dominio.
4. Por eso: **su propio eval set** + un harness de **varias dimensiones** (S06 = entrega M2).

*SI4006 · Universidad EAFIT · Sesión 5 — Métricas de evaluación  ·  SOLUCIONES.*